# Phase 3 V1: PEFT experiments with LoRA, LoRMA+, and BitFit

This is a **training notebook**, separate from the short ensemble notebook. It starts from the supplied Phase 1 V1 checkpoint and trains three parameter-efficient Phase 3 models on the labeled rhetorical-role data. It does not retrain Phase 1 language modeling.

The experiments use the same V1 tokenizer, model architecture, train/dev split, sentence representation, loss, and development selection metric. Only the adaptation method changes. Each method is initialized independently from the same untouched V1 checkpoint.

The three methods are:

1. **LoRA:** freezes the V1 weights and learns additive low-rank updates on the attention and feed-forward linear layers.
2. **LoRMA+:** freezes the V1 weights and learns the paper's multiplicative low-rank update with additive identity initialization: `(I + α/r BA) W₀`.
3. **BitFit:** freezes the V1 weights and updates only the existing transformer bias terms, alongside the task classifier.

The best method is selected using the mean of CL and IT development Macro-F1 with all 13 labels included. Test data is used only after selection. The notebook also saves each method's test predictions for transparent comparison. A higher development score does not guarantee a higher hidden-test score.


## Kaggle setup

Attach the dataset that contains the existing files `bpe_en_25k.json`, `v1_model.pt`, `label_vocab.json`, `CL_train.json`, `CL_dev.json`, `IT_train.json`, `IT_dev.json`, `CL_test.json`, and `IT_test.json`. The notebook performs strict V1 checkpoint loading and stops if a required asset is missing or the checkpoint does not match the supplied V1 architecture.

Select a Kaggle GPU accelerator for practical training time. This notebook runs three methods sequentially, with up to three epochs per method. It saves small trainable-parameter checkpoints, histories, development metrics, domain predictions, and a Codabench ZIP in `/kaggle/working/phase3_rr_v1_peft_lora_lorma_bitfit`.

The uploaded `v1_model.pt` is the existing Phase 1 checkpoint. It is loaded as the common starting point; Phase 1 pretraining is never started here.


In [ ]:
import os, json, math, random, time, copy, gc, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v1_peft_lora_lorma_bitfit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared PEFT experiment settings. The 512-token cap affects fewer than 0.1% of the supplied
# sentence examples under the exact V1 BPE tokenizer; the notebook reports the count.
MAX_SENTENCE_TOKENS = 512
BATCH_SIZE = 4
EPOCHS = 3
ADAPTER_LR = 2e-4
HEAD_LR = 5e-4
BITFIT_LR = 5e-4
LORA_RANK = 8
LORA_ALPHA = 16
ADAPTER_DROPOUT = 0.0
WEIGHT_DECAY = 0.01
PATIENCE = 2
HEAD_DROPOUT = 0.10
USE_CLASS_WEIGHTS = True
USE_MIXED_PRECISION = True
NUM_WORKERS = 0

print("Device:", DEVICE)
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Find each supplied asset under Kaggle's mounted input datasets.
INPUT_ROOT = Path("/kaggle/input")
ASSET_OVERRIDES = {
    # Optional explicit paths if Kaggle contains duplicate names.
    # Example: "v1_model.pt": "/kaggle/input/my-data/v1_model.pt"
}

def resolve_input_file(filename):
    if filename in ASSET_OVERRIDES:
        path = Path(ASSET_OVERRIDES[filename])
        if not path.is_file():
            raise FileNotFoundError(f"Configured path does not exist: {path}")
        return path
    matches = sorted(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename!r} under {INPUT_ROOT}; found {matches}. "
            "Set ASSET_OVERRIDES if duplicate copies are attached."
        )
    return matches[0]

ASSETS = {name: resolve_input_file(name) for name in (
    "bpe_en_25k.json", "v1_model.pt", "label_vocab.json",
    "CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json",
    "CL_test.json", "IT_test.json")}
for name, path in ASSETS.items():
    print(f"{name}: {path}")


## Original V1 tokenizer loading

The following tokenizer loading and tokenizer implementation cells are copied verbatim from original V1 notebook cells 13 and 14. The original tokenizer uses the `itos`/`merges` payload and word-by-word byte-level BPE.

In [ ]:
# Load the exact 25k BPE tokenizer asset resolved above.
BPE_PATH = ASSETS["bpe_en_25k.json"]
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)
bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]
assert bpe_vocab_size == 25_000 and len(bpe_itos) == 25_000
print("Tokenizer:", BPE_PATH, "| vocab:", bpe_vocab_size, "| merges:", len(bpe_merges))

bpe_stoi = {token: idx for idx, token in enumerate(bpe_itos)}
merge_to_id = {}
for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge
    merge_to_id[(bpe_stoi[left_token], bpe_stoi[right_token])] = 261 + merge_index

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]


In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

In [ ]:
# Load and audit the Phase 3 label vocabulary and data schemas.
LABEL_VOCAB_PATH = ASSETS["label_vocab.json"]
with open(LABEL_VOCAB_PATH, "r", encoding="utf-8") as f: label_to_id = json.load(f)
label_to_id = {str(k): int(v) for k, v in label_to_id.items()}
assert len(label_to_id) == 13, f"Expected 13 labels, found {len(label_to_id)}"
assert sorted(label_to_id.values()) == list(range(13)), "label_vocab.json indices must be contiguous 0..12"
id_to_label = {i: label for label, i in label_to_id.items()}
label_set = set(label_to_id)
print("Official label mapping:", label_to_id)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def audit_labeled(path, split_name):
    records = load_json(path)
    if not isinstance(records, list): raise TypeError(f"{split_name}: expected a list of documents")
    ids=[]; sentence_count=0; empty_count=0; counts=Counter()
    for row_num, row in enumerate(records):
        if not isinstance(row, dict): raise TypeError(f"{split_name} row {row_num}: expected object")
        for key in ("id", "text", "labels"):
            if key not in row: raise ValueError(f"{split_name} row {row_num}: missing {key!r}")
        if not isinstance(row["text"], list) or not isinstance(row["labels"], list):
            raise TypeError(f"{split_name} row {row_num}: text and labels must be lists")
        if len(row["text"]) != len(row["labels"]):
            raise ValueError(f"{split_name} {row['id']}: {len(row['text'])} sentences but {len(row['labels'])} labels")
        ids.append(str(row["id"])); sentence_count += len(row["text"])
        empty_count += sum(not str(x).strip() for x in row["text"])
        unknown=set(row["labels"])-label_set
        if unknown: raise ValueError(f"{split_name} {row['id']}: unknown labels {sorted(unknown)}")
        counts.update(row["labels"])
    if len(ids) != len(set(ids)): raise ValueError(f"{split_name}: duplicate document IDs")
    print(f"{split_name}: {len(records)} documents, {sentence_count} sentences, {empty_count} empty sentences")
    if counts: print("  label counts:", dict(counts))
    return records, set(ids)

CL_TRAIN, cl_train_ids = audit_labeled(ASSETS["CL_train.json"], "CL train")
CL_DEV, cl_dev_ids = audit_labeled(ASSETS["CL_dev.json"], "CL dev")
IT_TRAIN, it_train_ids = audit_labeled(ASSETS["IT_train.json"], "IT train")
IT_DEV, it_dev_ids = audit_labeled(ASSETS["IT_dev.json"], "IT dev")
assert not (cl_train_ids & cl_dev_ids), "CL train/dev document leakage"
assert not (it_train_ids & it_dev_ids), "IT train/dev document leakage"

def load_test_docs(path, domain):
    data = load_json(path)
    if not isinstance(data, dict): raise TypeError(f"{domain} test: expected mapping document ID -> sentence list")
    docs=[]
    for doc_id, sentences in data.items():
        if not isinstance(sentences, list): raise TypeError(f"{domain} test {doc_id}: expected sentence list")
        docs.append((str(doc_id), [str(s) for s in sentences]))
    print(f"{domain} test: {len(docs)} documents, {sum(len(x[1]) for x in docs)} sentences")
    return docs

CL_TEST = load_test_docs(ASSETS["CL_test.json"], "CL")
IT_TEST = load_test_docs(ASSETS["IT_test.json"], "IT")

In [ ]:
# Tokenize examples once. Match the original V1 pretraining convention: no BOS/EOS.
def encode_sentence(sentence):
    ids = tokenizer.encode(str(sentence), add_bos=False, add_eos=False)
    if not ids: ids = [UNK_TOKEN_ID]
    truncated = len(ids) > MAX_SENTENCE_TOKENS
    if truncated: ids = ids[:MAX_SENTENCE_TOKENS]
    return ids, truncated

def make_labeled_examples(records, domain):
    out=[]; n_truncated=0
    for doc in records:
        for sent_idx, (sentence, label) in enumerate(zip(doc["text"], doc["labels"])):
            ids, truncated = encode_sentence(sentence); n_truncated += int(truncated)
            out.append({"input_ids":ids,"label":label_to_id[label],"domain":domain,
                        "doc_id":str(doc["id"]),"sentence_index":sent_idx})
    return out, n_truncated

CL_TRAIN_EX, cl_trunc = make_labeled_examples(CL_TRAIN,"CL")
IT_TRAIN_EX, it_trunc = make_labeled_examples(IT_TRAIN,"IT")
CL_DEV_EX, cl_dev_trunc = make_labeled_examples(CL_DEV,"CL")
IT_DEV_EX, it_dev_trunc = make_labeled_examples(IT_DEV,"IT")
TRAIN_EX = CL_TRAIN_EX + IT_TRAIN_EX
DEV_BY_DOMAIN = {"CL":CL_DEV_EX,"IT":IT_DEV_EX}
print("Sentence examples: train",len(TRAIN_EX),"CL dev",len(CL_DEV_EX),"IT dev",len(IT_DEV_EX))
print("Truncated examples at",MAX_SENTENCE_TOKENS,"tokens:",
      {"CL train":cl_trunc,"IT train":it_trunc,"CL dev":cl_dev_trunc,"IT dev":it_dev_trunc})
print("Training class counts:", {id_to_label[i]:sum(x["label"]==i for x in TRAIN_EX) for i in range(13)})

class RRSentenceDataset(Dataset):
    def __init__(self, examples): self.examples=examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

def collate_labeled(rows):
    seqs=[torch.tensor(r["input_ids"],dtype=torch.long) for r in rows]
    lengths=torch.tensor([len(x) for x in seqs],dtype=torch.long)
    input_ids=pad_sequence(seqs,batch_first=True,padding_value=PAD_TOKEN_ID)
    labels=torch.tensor([r["label"] for r in rows],dtype=torch.long)
    return input_ids,lengths,labels

TRAIN_LOADER=DataLoader(RRSentenceDataset(TRAIN_EX),batch_size=BATCH_SIZE,shuffle=True,
                        num_workers=NUM_WORKERS,collate_fn=collate_labeled,pin_memory=DEVICE.type=="cuda")
DEV_LOADERS={d:DataLoader(RRSentenceDataset(ex),batch_size=BATCH_SIZE,shuffle=False,
                           num_workers=NUM_WORKERS,collate_fn=collate_labeled,pin_memory=DEVICE.type=="cuda")
             for d,ex in DEV_BY_DOMAIN.items()}

## Original V1 architecture

This is original V1 notebook cell 26 copied verbatim. It defines the pretrained V1 module names and parameter shapes. The following Phase 3 cells instantiate this exact class from the checkpoint configuration and load its saved weights strictly.

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
# Load the already-trained V1 checkpoint. This does not execute Phase 1 training.
CHECKPOINT_PATH = resolve_input_file("v1_model.pt")
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
    raise ValueError("Expected the supplied V1 checkpoint wrapper with model_state_dict")
model_config = checkpoint.get("config", {})
expected_config = {"vocab_size":25000,"n_embd":768,"n_layer":12,"n_head":12,
                   "n_kv_head":4,"ffn_multiplier":4,"context_length":1024}
for key, expected in expected_config.items():
    if key in model_config and int(model_config[key]) != expected:
        raise ValueError(f"Checkpoint config mismatch for {key}: {model_config[key]} vs {expected}")

backbone = V1LegalLM(
    vocab_size=int(model_config.get("vocab_size",25000)),
    n_embd=int(model_config.get("n_embd",768)),
    n_layer=int(model_config.get("n_layer",12)),
    n_head=int(model_config.get("n_head",12)),
    n_kv_head=int(model_config.get("n_kv_head",4)),
    max_seq_len=int(model_config.get("context_length",1024)),
    ffn_multiplier=int(model_config.get("ffn_multiplier",4)),
    dropout=float(model_config.get("dropout",0.0)))
load_result = backbone.load_state_dict(checkpoint["model_state_dict"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
backbone = backbone.to(DEVICE)
print("Strict V1 checkpoint load succeeded.")
print("Checkpoint:", CHECKPOINT_PATH)
print("Best Phase 1 validation loss:", checkpoint.get("best_val_loss"))
print("Backbone parameters:", f"{sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# The hidden-state extraction below follows the exact V1 forward path,
# omitting only the 25,000-way language-model output projection.
def encode_v1_hidden(v1_model, input_ids):
    x = v1_model.token_embedding(input_ids)
    for block in v1_model.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)
    return v1_model.final_norm(x)

probe_ids = torch.tensor([[4, 5, 6, 7, 8]], dtype=torch.long, device=DEVICE)
backbone.eval()
with torch.no_grad():
    full_logits, _ = backbone(probe_ids, use_cache=False)
    hidden = encode_v1_hidden(backbone, probe_ids)
    projected_logits = backbone.lm_head(hidden)
assert torch.allclose(full_logits, projected_logits, atol=1e-5, rtol=1e-5)
print("V1 hidden extraction matches the original forward path; max difference:",
      float((full_logits - projected_logits).abs().max()))

class LowRankLinear(nn.Module):
    # LoRA or LoRMA+ wrapper for one frozen linear projection.
    def __init__(self, base, method, rank=LORA_RANK, alpha=LORA_ALPHA,
                 dropout=ADAPTER_DROPOUT):
        super().__init__()
        if method not in {"lora", "lorma_plus"}:
            raise ValueError(f"Unsupported low-rank method: {method}")
        self.base = base
        self.method = method
        self.rank = int(rank)
        self.scale = float(alpha) / self.rank
        self.adapter_dropout = nn.Dropout(float(dropout))
        for parameter in self.base.parameters():
            parameter.requires_grad = False

        if method == "lora":
            # Additive update: W0 x + (alpha/r) B A x.
            self.adapter_A = nn.Linear(base.in_features, self.rank, bias=False)
            self.adapter_B = nn.Linear(self.rank, base.out_features, bias=False)
        else:
            # LoRMA+ update: (I + (alpha/r) B A) W0 x.
            # A and B act on the frozen projection's output space.
            self.adapter_A = nn.Linear(base.out_features, self.rank, bias=False)
            self.adapter_B = nn.Linear(self.rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.adapter_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.adapter_B.weight)

    def forward(self, x):
        base_output = self.base(x)
        if self.method == "lora":
            update = self.adapter_B(self.adapter_dropout(self.adapter_A(x)))
        else:
            update = self.adapter_B(self.adapter_dropout(self.adapter_A(base_output)))
        return base_output + self.scale * update

class V1RhetoricalRolePredictor(nn.Module):
    def __init__(self, pretrained_backbone, n_labels=13, dropout=0.10):
        super().__init__()
        self.backbone = pretrained_backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.backbone.n_embd, n_labels)

    def forward(self, input_ids, lengths):
        hidden = encode_v1_hidden(self.backbone, input_ids)
        rows = torch.arange(input_ids.shape[0], device=input_ids.device)
        sentence_state = hidden[rows, lengths.to(input_ids.device) - 1]
        return self.classifier(self.dropout(sentence_state))

TARGET_LINEAR_NAMES = (
    "q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"
)

def load_fresh_v1_backbone():
    # Build and strict-load a fresh copy for an identical V1 initialization per method.
    model = V1LegalLM(
        vocab_size=int(model_config.get("vocab_size", 25000)),
        n_embd=int(model_config.get("n_embd", 768)),
        n_layer=int(model_config.get("n_layer", 12)),
        n_head=int(model_config.get("n_head", 12)),
        n_kv_head=int(model_config.get("n_kv_head", 4)),
        max_seq_len=int(model_config.get("context_length", 1024)),
        ffn_multiplier=int(model_config.get("ffn_multiplier", 4)),
        dropout=float(model_config.get("dropout", 0.0)))
    result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    assert not result.missing_keys and not result.unexpected_keys
    return model

def make_peft_model(method):
    backbone = load_fresh_v1_backbone()
    for parameter in backbone.parameters():
        parameter.requires_grad = False

    if method in {"lora", "lorma_plus"}:
        for block in backbone.blocks:
            for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
                original = getattr(block.attention, name)
                setattr(block.attention, name, LowRankLinear(original, method))
            for name in ("fc1", "fc2"):
                original = getattr(block.ffn, name)
                setattr(block.ffn, name, LowRankLinear(original, method))
    elif method == "bitfit":
        # BitFit changes existing transformer biases only; the task head is also trained.
        for name, parameter in backbone.named_parameters():
            if name.startswith("blocks.") and name.endswith(".bias"):
                parameter.requires_grad = True
    else:
        raise ValueError(f"Unknown adaptation method: {method}")

    # Give all runs the same random initialization for the task classifier.
    torch.manual_seed(SEED + 1701)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 1701)
    model = V1RhetoricalRolePredictor(backbone, len(label_to_id), HEAD_DROPOUT)
    return model.to(DEVICE)

def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters()
                    if parameter.requires_grad)
    return {"total": int(total), "trainable": int(trainable),
            "trainable_percent": float(100.0 * trainable / max(total, 1))}

# The original full model is no longer needed after the equivalence probe.
del backbone
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Fixed 13-label metrics: absent classes receive zero F1 in the macro average.
def metrics_from_ids(gold, pred):
    cm = np.zeros((13, 13), dtype=np.int64)
    for y, p in zip(gold, pred):
        cm[int(y), int(p)] += 1
    tp = np.diag(cm).astype(float)
    support = cm.sum(axis=1).astype(float)
    predicted = cm.sum(axis=0).astype(float)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros_like(tp), where=(precision + recall) > 0)
    per_class = {id_to_label[i]: {"precision": float(precision[i]),
                                  "recall": float(recall[i]), "f1": float(f1[i]),
                                  "support": int(support[i])} for i in range(13)}
    return {"macro_precision": float(precision.mean()),
            "macro_recall": float(recall.mean()), "macro_f1": float(f1.mean()),
            "per_class": per_class, "confusion_matrix": cm.tolist()}

def amp_context():
    if DEVICE.type != "cuda" or not USE_MIXED_PRECISION:
        return nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype)

@torch.inference_mode()
def evaluate_loader(model, loader):
    model.eval()
    gold, pred = [], []
    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        with amp_context():
            logits = model(input_ids, lengths)
        gold.extend(labels.tolist())
        pred.extend(logits.argmax(dim=-1).cpu().tolist())
    return metrics_from_ids(gold, pred)


In [ ]:
# Train-only inverse-square-root weights are shared by all three controlled runs.
if USE_CLASS_WEIGHTS:
    counts = np.bincount([row["label"] for row in TRAIN_EX], minlength=13).astype(float)
    weights = np.zeros(13, dtype=np.float32)
    present = counts > 0
    weights[present] = 1.0 / np.sqrt(counts[present])
    weights[present] /= weights[present].mean()
    weights[present] = np.clip(weights[present], 0.25, 5.0)
    CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    print("Train-only class weights:", weights.tolist())
else:
    CLASS_WEIGHTS = None
    criterion = nn.CrossEntropyLoss()

METHODS = ("lora", "lorma_plus", "bitfit")
PEFT_RESULTS = {}
PEFT_HISTORY = {}
PEFT_CHECKPOINTS = {}
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

for method in METHODS:
    run_seed = SEED
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(run_seed)

    rr_model = make_peft_model(method)
    # Keep minibatch order and classifier-dropout RNG aligned across methods.
    torch.manual_seed(run_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(run_seed)
    params = count_parameters(rr_model)
    trainable_named = [(name, p) for name, p in rr_model.named_parameters()
                       if p.requires_grad]
    assert trainable_named, f"{method}: no trainable parameters"
    if method == "bitfit":
        adapter_params = [p for name, p in trainable_named if not name.startswith("classifier.")]
        head_params = list(rr_model.classifier.parameters())
        param_groups = [{"params": adapter_params, "lr": BITFIT_LR},
                        {"params": head_params, "lr": HEAD_LR}]
    else:
        adapter_params = [p for name, p in trainable_named if not name.startswith("classifier.")]
        head_params = list(rr_model.classifier.parameters())
        param_groups = [{"params": adapter_params, "lr": ADAPTER_LR},
                        {"params": head_params, "lr": HEAD_LR}]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    amp_enabled = DEVICE.type == "cuda" and USE_MIXED_PRECISION
    amp_dtype = torch.bfloat16 if amp_enabled and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled and amp_dtype == torch.float16)

    history = []
    best_score = -1.0
    best_epoch = None
    best_metrics = None
    best_trainable_state = None
    stale_epochs = 0
    run_start = time.time()
    for epoch in range(1, EPOCHS + 1):
        torch.manual_seed(SEED + epoch)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED + epoch)
        rr_model.train()
        total_loss, total_examples = 0.0, 0
        epoch_start = time.time()
        for input_ids, lengths, labels in TRAIN_LOADER:
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            lengths = lengths.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = rr_model(input_ids, lengths)
                loss = criterion(logits.float(), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"{method}: non-finite loss")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for _, p in trainable_named], MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach()) * len(labels)
            total_examples += len(labels)

        domain_metrics = {domain: evaluate_loader(rr_model, DEV_LOADERS[domain])
                          for domain in ("CL", "IT")}
        score = (domain_metrics["CL"]["macro_f1"] +
                 domain_metrics["IT"]["macro_f1"]) / 2.0
        row = {"epoch": epoch,
               "train_loss": total_loss / max(total_examples, 1),
               "dev_CL_macro_f1": domain_metrics["CL"]["macro_f1"],
               "dev_IT_macro_f1": domain_metrics["IT"]["macro_f1"],
               "dev_mean_domain_macro_f1": score,
               "seconds": time.time() - epoch_start}
        history.append(row)
        print(f"{method} | epoch {epoch}/{EPOCHS} | loss {row['train_loss']:.4f} | "
              f"CL {row['dev_CL_macro_f1']:.4f} | IT {row['dev_IT_macro_f1']:.4f} | "
              f"mean {score:.4f}")

        if score > best_score:
            best_score, best_epoch, best_metrics = score, epoch, domain_metrics
            best_trainable_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in rr_model.named_parameters()
                if parameter.requires_grad
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                print(f"{method}: early stopping after {epoch} epochs")
                break

    checkpoint_path = OUTPUT_DIR / f"{method}_adapter.pt"
    torch.save({"method": method, "trainable_state_dict": best_trainable_state,
                "best_epoch": best_epoch, "dev_metrics": best_metrics,
                "best_dev_mean_domain_macro_f1": best_score,
                "label_to_id": label_to_id, "id_to_label": id_to_label,
                "v1_config": model_config,
                "training_config": {"seed": run_seed, "rank": LORA_RANK,
                    "alpha": LORA_ALPHA, "epochs_max": EPOCHS,
                    "batch_size": BATCH_SIZE, "max_sentence_tokens": MAX_SENTENCE_TOKENS,
                    "class_weights": None if CLASS_WEIGHTS is None else CLASS_WEIGHTS.cpu().tolist(),
                    "adapter_lr": ADAPTER_LR, "head_lr": HEAD_LR,
                    "bitfit_lr": BITFIT_LR}}, checkpoint_path)
    PEFT_RESULTS[method] = {
        "best_epoch": best_epoch,
        "best_dev_mean_domain_macro_f1": best_score,
        "domains": best_metrics,
        "parameters": params,
        "training_seconds": time.time() - run_start,
        "checkpoint": str(checkpoint_path),
        "checkpoint_bytes": checkpoint_path.stat().st_size,
    }
    PEFT_HISTORY[method] = history
    PEFT_CHECKPOINTS[method] = checkpoint_path
    print(f"{method}: trainable {params['trainable']:,}/{params['total']:,} "
          f"({params['trainable_percent']:.4f}%), best epoch {best_epoch}, "
          f"dev mean {best_score:.4f}, saved {checkpoint_path}")

    del rr_model, optimizer, scaler, best_trainable_state
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

selected_method = max(METHODS, key=lambda name: PEFT_RESULTS[name]["best_dev_mean_domain_macro_f1"])
comparison = {"selected_method_by_dev": selected_method,
              "selection_rule": "highest mean fixed-13-label Macro-F1 across CL and IT dev",
              "methods": PEFT_RESULTS,
              "important_note": "Development selection does not guarantee a higher hidden-test score."}
with open(OUTPUT_DIR / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(PEFT_HISTORY, f, ensure_ascii=False, indent=2)
print("Selected by development data only:", selected_method)


In [ ]:
# Rebuild each selected PEFT model from the exact same Phase 1 checkpoint,
# load only its trainable parameters, and generate aligned test predictions.
def load_peft_weights(model, checkpoint_path):
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    trainable = {name: parameter for name, parameter in model.named_parameters()
                 if parameter.requires_grad}
    state = saved["trainable_state_dict"]
    if set(trainable) != set(state):
        missing = sorted(set(trainable) - set(state))
        extra = sorted(set(state) - set(trainable))
        raise ValueError(f"Trainable parameter mismatch. Missing={missing}; extra={extra}")
    with torch.no_grad():
        for name, parameter in trainable.items():
            if parameter.shape != state[name].shape:
                raise ValueError(f"{name}: adapter shape mismatch")
            parameter.copy_(state[name].to(device=parameter.device, dtype=parameter.dtype))
    return saved

def infer_test_domain(model, domain, docs):
    examples = []
    expected = {}
    for doc_id, sentences in docs:
        key = str(doc_id)
        if key in expected:
            raise ValueError(f"{domain}: duplicate test document ID {key}")
        expected[key] = len(sentences)
        for sent_idx, sentence in enumerate(sentences):
            ids, truncated = encode_sentence(sentence)
            examples.append({"input_ids": ids, "doc_id": key,
                             "sentence_index": sent_idx, "truncated": truncated})

    def collate(rows):
        seqs = [torch.tensor(row["input_ids"], dtype=torch.long) for row in rows]
        lengths = torch.tensor([len(seq) for seq in seqs], dtype=torch.long)
        return pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN_ID), lengths

    loader = DataLoader(examples, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate)
    grouped = defaultdict(list)
    example_pos = 0
    truncated_count = 0
    model.eval()
    with torch.inference_mode():
        for input_ids, lengths in loader:
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            lengths = lengths.to(DEVICE, non_blocking=True)
            with amp_context():
                logits = model(input_ids, lengths)
            for pred_id in logits.argmax(dim=-1).cpu().tolist():
                meta = examples[example_pos]
                example_pos += 1
                grouped[meta["doc_id"]].append((meta["sentence_index"],
                                                  id_to_label[int(pred_id)]))
                truncated_count += int(meta["truncated"])

    output = []
    for doc_id, sentences in docs:
        key = str(doc_id)
        ordered = [label for _, label in sorted(grouped[key])]
        if len(ordered) != expected[key] or len(ordered) != len(sentences):
            raise ValueError(f"{domain}/{key}: output count differs from sentence count")
        if any(label not in label_set for label in ordered):
            raise ValueError(f"{domain}/{key}: unknown output label")
        output.append({"id": doc_id, "predictions": ordered})
    if example_pos != sum(expected.values()) or len(output) != len(docs):
        raise ValueError(f"{domain}: inference did not cover every test sentence")
    return output, truncated_count

ALL_TEST_PREDICTIONS = {}
for method in METHODS:
    rr_model = make_peft_model(method)
    saved = load_peft_weights(rr_model, PEFT_CHECKPOINTS[method])
    method_outputs = {}
    for domain, docs in (("CL", CL_TEST), ("IT", IT_TEST)):
        rows, n_truncated = infer_test_domain(rr_model, domain, docs)
        method_outputs[domain] = rows
        path = OUTPUT_DIR / f"{method}_{domain}_test_predictions.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"domain": domain, "method": method, "documents": rows},
                      f, ensure_ascii=False, indent=2)
        print(f"{method}/{domain}: {len(rows)} docs, "
              f"{sum(len(row['predictions']) for row in rows)} sentences, "
              f"truncated {n_truncated}; {path}")
    ALL_TEST_PREDICTIONS[method] = method_outputs
    del rr_model, saved
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

selected_predictions = ALL_TEST_PREDICTIONS[selected_method]
submission = {}
for domain in ("CL", "IT"):
    expected_docs = CL_TEST if domain == "CL" else IT_TEST
    rows = selected_predictions[domain]
    if [str(row["id"]) for row in rows] != [str(doc_id) for doc_id, _ in expected_docs]:
        raise ValueError(f"{domain}: prediction document IDs/order differ from test input")
    for row, (doc_id, sentences) in zip(rows, expected_docs):
        key = str(row["id"])
        if key in submission:
            raise ValueError(f"Duplicate submission document ID: {key}")
        if len(row["predictions"]) != len(sentences):
            raise ValueError(f"{domain}/{key}: submission sentence count mismatch")
        submission[key] = row["predictions"]

prediction_path = OUTPUT_DIR / "prediction.json"
with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "selected_method.json", "w", encoding="utf-8") as f:
    json.dump({"selected_method": selected_method,
               "selection_score": PEFT_RESULTS[selected_method]["best_dev_mean_domain_macro_f1"],
               "selection_rule": comparison["selection_rule"]},
              f, ensure_ascii=False, indent=2)

zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(prediction_path, arcname="prediction.json")
with zipfile.ZipFile(zip_path, "r") as archive:
    if archive.namelist() != ["prediction.json"]:
        raise ValueError(f"Unexpected ZIP members: {archive.namelist()}")
    if archive.testzip() is not None:
        raise ValueError("Submission ZIP CRC check failed")
    packaged = json.loads(archive.read("prediction.json").decode("utf-8"))
if packaged != submission:
    raise ValueError("ZIP payload does not match prediction.json")
print("Selected method:", selected_method)
print("Submission ZIP:", zip_path)
print("PASS: valid root-level prediction.json, document coverage, sentence counts, and labels.")


## What this experiment tells us

The output folder contains a small trainable-parameter checkpoint and full development metrics for each method, training history, trainable/total parameter counts, CL/IT test prediction files for each method, and one `submission.zip` selected using development Macro-F1 only.

This is a V1-backbone PEFT comparison. It does not implement the separate knowledge-distillation experiment or the other Phase 1 backbones. The saved project plan records those as additional assignment requirements; the official assignment brief should be checked before claiming the full assignment is complete.

No PEFT method is guaranteed to beat full fine-tuning or the current 0.4311/0.4292 Codabench results. Compare the eventual hidden-test score and preserve the previous best submission if the new one is lower.


## Research references

1. Hu, E. J. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* https://arxiv.org/abs/2106.09685
2. Bihany, H., Patel, S., and Modi, A. (2025). *LoRMA: Low-Rank Multiplicative Adaptation for LLMs.* Findings of ACL 2025, pp. 10113–10133. https://aclanthology.org/2025.findings-acl.527/
3. Ben Zaken, E., Goldberg, Y., and Ravfogel, S. (2022). *BitFit: Simple Parameter-efficient Fine-tuning for Transformer-based Masked Language-models.* ACL 2022. https://aclanthology.org/2022.acl-short.1/

LoRA and LoRMA are separate methods: LoRA adds a low-rank weight update; LoRMA+ applies a low-rank multiplicative transformation with identity initialization. BitFit updates existing bias terms only. All three freeze most of the pretrained V1 weights and therefore fall under the broader PEFT category.
